# Test Your Algorithm

## Instructions
1. From the **Pulse Rate Algorithm** Notebook you can do one of the following:
   - Copy over all the **Code** section to the following Code block.
   - Download as a Python (`.py`) and copy the code to the following Code block.
2. In the bottom right, click the <span style="color:blue">Test Run</span> button. 

### Didn't Pass
If your code didn't pass the test, go back to the previous Concept or to your local setup and continue iterating on your algorithm and try to bring your training error down before testing again.

### Pass
If your code passes the test, complete the following! You **must** include a screenshot of your code and the Test being **Passed**. Here is what the starter filler code looks like when the test is run and should be similar. A passed test will include in the notebook a green outline plus a box with **Test passed:** and in the Results bar at the bottom the progress bar will be at 100% plus a checkmark with **All cells passed**.
![Example](example.png)

1. Take a screenshot of your code passing the test, make sure it is in the format `.png`. If not a `.png` image, you will have to edit the Markdown render the image after Step 3. Here is an example of what the `passed.png` would look like 
2. Upload the screenshot to the same folder or directory as this jupyter notebook.
3. Rename the screenshot to `passed.png` and it should show up below.
![Passed](passed.png)
4. Download this jupyter notebook as a `.pdf` file. 
5. Continue to Part 2 of the Project. 

In [2]:
import numpy as np
import scipy as sp
import scipy.io
from scipy import signal


def _bandpass_filter(x, fs, low_cut, high_cut, order=3):
    nyq = fs / 2.0
    low = max(low_cut / nyq, 1e-4)
    high = min(high_cut / nyq, 0.999)
    if high <= low:
        high = min(low + 0.1, 0.999)
    b, a = signal.butter(order, [low, high], btype="band")
    return signal.filtfilt(b, a, x)


def _estimate_window_bpm(ppg_win, acc_mag_win, fs, params, prev_bpm=None):
    ppg_f = _bandpass_filter(ppg_win, fs, params["low_cut_hz"], params["high_cut_hz"])
    acc_f = _bandpass_filter(acc_mag_win, fs, 0.4, 8.0)

    n_raw = len(ppg_f)
    n_fft = int(2 ** np.ceil(np.log2(n_raw)) * params["fft_pad"])

    ppg_spec = np.abs(np.fft.rfft(ppg_f * np.hanning(n_raw), n=n_fft)) ** 2
    acc_spec = np.abs(np.fft.rfft(acc_f * np.hanning(n_raw), n=n_fft)) ** 2
    freqs = np.fft.rfftfreq(n_fft, d=1.0 / fs)

    hr_low = params.get("hr_low_bpm", 40.0) / 60.0
    hr_high = params.get("hr_high_bpm", 240.0) / 60.0
    band = (freqs >= hr_low) & (freqs <= hr_high)
    if not np.any(band):
        return 75.0, 0.0

    f_band = freqs[band]
    p_band = ppg_spec[band].copy()
    a_band = acc_spec[band]

    a_floor = np.median(a_band) + 1e-8
    a_norm = a_band / a_floor
    p_floor = np.median(p_band) + 1e-8
    p_norm = p_band / p_floor

    width_hz = params["motion_width_bpm"] / 60.0
    motion_idx = np.where(a_norm > params["motion_gate"])[0]
    for idx in motion_idx:
        mf = f_band[idx]
        near = np.abs(f_band - mf) <= width_hz
        p_norm[near] *= params["motion_suppress"]

    score = p_norm / (1.0 + params["motion_penalty_alpha"] * a_norm)

    if prev_bpm is not None and np.isfinite(prev_bpm):
        bpm_band = f_band * 60.0
        sigma = max(params["continuity_sigma_bpm"], 1.0)
        continuity = np.exp(-0.5 * ((bpm_band - prev_bpm) / sigma) ** 2)
        score *= (1.0 + params["continuity_weight"] * continuity)

    peak_idx = int(np.argmax(score))

    motion_peak_idx = int(np.argmax(a_band))
    motion_bpm = f_band[motion_peak_idx] * 60.0
    best_bpm = f_band[peak_idx] * 60.0

    if abs(best_bpm - motion_bpm) <= params["motion_conflict_bpm"]:
        candidate_idx = np.argsort(score)[::-1][: min(8, len(score))]
        best_alt_idx = peak_idx
        best_alt_score = score[peak_idx]
        tol = params["harmonic_tolerance_bpm"]
        for idx in candidate_idx:
            cand_bpm = f_band[idx] * 60.0
            harmonic_match = (
                abs(cand_bpm - 2.0 * motion_bpm) <= tol
                or abs(cand_bpm - 0.5 * motion_bpm) <= tol
            )
            if harmonic_match:
                boosted = score[idx] * (1.0 + params["harmonic_bonus"])
                if boosted > best_alt_score:
                    best_alt_idx = int(idx)
                    best_alt_score = float(boosted)
        peak_idx = best_alt_idx

    if len(score) > 1:
        top2 = np.partition(score, -2)[-2:]
        second_best = float(top2[0] if top2[1] == score[peak_idx] else top2[1])
    else:
        second_best = 0.0

    peak_score = float(score[peak_idx]) + 1e-12
    bpm = float(f_band[peak_idx] * 60.0)

    local_width_hz = params["confidence_local_width_bpm"] / 60.0
    near_peak = np.abs(f_band - f_band[peak_idx]) <= local_width_hz
    concentration = float((np.sum(p_band[near_peak]) + 1e-12) / (np.sum(p_band) + 1e-12))

    peak_ratio = float(peak_score / (second_best + 1e-12))
    sep = np.tanh(0.6 * max(0.0, peak_ratio - 1.0))
    motion_pen = float(a_norm[peak_idx] / (np.max(a_norm) + 1e-12))

    confidence = concentration * (0.4 + 0.6 * sep) * (1.0 - 0.5 * motion_pen)
    confidence = float(np.clip(confidence, 0.0, 1.0))

    return bpm, confidence


BEST_PARAMS = {
    "window_sec": 11,
    "step_sec": 2,
    "low_cut_hz": 1.5255647229607623,
    "high_cut_hz": 3.242254383158851,
    "fft_pad": 4,
    "motion_gate": 3.1572021999498316,
    "motion_width_bpm": 3.437736016078344,
    "motion_suppress": 0.9313933905050024,
    "motion_penalty_alpha": 0.5349392101047147,
    "peak_prominence": 0.28051598726525134,
    "min_quality": 0.010297782199481126,
    "confidence_gamma": 2.7291486627490706,
    "hr_low_bpm": 40.0,
    "hr_high_bpm": 240.0,
    "continuity_sigma_bpm": 30.0,
    "continuity_weight": 1.4085762864807632,
    "continuity_blend_low_conf": 0.46910435944882783,
    "continuity_blend_high_conf": 0.01698459140639261,
    "adaptive_low_conf": 0.019578510057082377,
    "adaptive_high_conf": 0.3476831085520954,
    "motion_conflict_bpm": 1.857805194720296,
    "harmonic_tolerance_bpm": 12.0,
    "harmonic_bonus": 0.5722832376262514,
    "confidence_local_width_bpm": 3.91831649764372,
}


def RunPulseRateAlgorithm(data_fl, ref_fl):
    """Estimate pulse-rate error and confidence for one trial."""
    params = BEST_PARAMS

    data = sp.io.loadmat(data_fl)["sig"]
    ppg, accx, accy, accz = data[2:]
    ref = sp.io.loadmat(ref_fl)["BPM0"].squeeze().astype(float)

    fs = 125.0
    window_len = int(params["window_sec"] * fs)
    step_len = int(params["step_sec"] * fs)

    if len(ppg) < window_len or step_len <= 0:
        return np.array([]), np.array([])

    acc_mag = np.sqrt(accx**2 + accy**2 + accz**2)

    possible = 1 + (len(ppg) - window_len) // step_len
    n_eval = min(len(ref), possible)

    est_bpm = []
    conf = []
    prev_bpm = None

    for i in range(n_eval):
        start = i * step_len
        stop = start + window_len
        ppg_win = ppg[start:stop]
        acc_win = acc_mag[start:stop]

        bpm_hat, confidence = _estimate_window_bpm(ppg_win, acc_win, fs, params, prev_bpm=prev_bpm)

        if prev_bpm is not None and np.isfinite(prev_bpm):
            low_c = params["adaptive_low_conf"]
            high_c = max(low_c + 1e-6, params["adaptive_high_conf"])
            if confidence <= low_c:
                blend = params["continuity_blend_low_conf"]
            elif confidence >= high_c:
                blend = params["continuity_blend_high_conf"]
            else:
                alpha = (confidence - low_c) / (high_c - low_c)
                blend = (1.0 - alpha) * params["continuity_blend_low_conf"] + alpha * params["continuity_blend_high_conf"]
            bpm_hat = (1.0 - blend) * bpm_hat + blend * prev_bpm

        est_bpm.append(bpm_hat)
        conf.append(confidence)

        if confidence >= params["min_quality"]:
            prev_bpm = bpm_hat

    est_bpm = np.asarray(est_bpm)
    conf = np.asarray(conf)
    errors = np.abs(est_bpm - ref[:n_eval])
    return errors, conf